# 셀1: 필요한 패키지 설치
- 주요 라이브러리 설치를 위한 셀
- MONAI, nibabel, tqdm, matplotlib 등 swin UNETR 및 의료 영상 처리에 필요한 패키지를 설치
- transformers는 Swin Transformer 관련 백본 모델 활용을 위한 라이브러리

In [1]:
# [MONAI, nibabel, tqdm]이 이미 설치되어 있는지 확인하고, 설치 안 되어 있으면 설치함
# - modai: 의료 영상 처리를 위한 PyTorch 기반 프레임워크
# - nibabel: .nii, .nill.gz 같은 의료 영상 포맷을 다루기 위한 라이브러리
# - tqdm: 진행률(progress bar) 표시를 위한 라이브러리
!python -c "import monai; import nibabel; import tqdm" || pip install -q "monai-weekly[nibabel, tqdm]"

# matplotlib이 설치되어 있는지 확인하고, 설치 안되어 있으면 설치함
# - matplotlib: 데이터 시각화를 위한 가장 기본적인 파이썬 라이브러리
!python -c "import matplotlib" || pip install -q matplotlib

# Swin UNETR 학습에 필요한 기타 필수 라이브러리 설치
!pip install scikit-image scikit-learn gdown torchvision pandas einops transformers

# 설명:
# - scikit-image: 이미지 처리 유틸리티 (예: 필터링, 마스크 등)
# - scikit-learn: 머신러닝 도구 (데이터셋 분할, 평가 등)
# - gdown: 구글 드라이브에서 파일을 쉽게 다운로드
# - torchvision: PyTorch에서 이미지 데이터 처리에 사용하는 모듈
# - pandas: 데이터 프레임 처리 (CSV 읽기 등)
# - transformers: Swin Transformer 사전 학습 모델을 불러올 때 사용

Traceback (most recent call last):
  File "<string>", line 1, in <module>
ModuleNotFoundError: No module named 'monai'

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 41.8 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 561.5/561.5 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [transformers] [transformers]ub]

[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


# 셀2: 주요 라이브러리 임포트 및 환경 출력
- 학습에 필요한 MONAI, PyTorch, 기타 유틸리티 라이브러리르 import
- print_config()를 통해 MONAI 환경 설정 정보를 출력 (버전, 디바이스 상태 등 확인 용)

In [2]:
# 🛠 기본 라이브러리
import os  # 파일 경로/이름 설정 및 디렉토리 작업용
import shutil  # 파일 및 폴더 복사/이동/삭제 등 유틸리티
import tempfile  # 임시 디렉토리 생성을 위한 모듈

# 📊 시각화 및 진행률 표시용
import matplotlib.pyplot as plt  # 이미지나 그래프 시각화를 위한 라이브러리
from tqdm import tqdm  # 반복문의 진행 상황을 보여주는 진행률 표시바

# 🧠 MONAI 관련 모듈 (의료 영상 AI를 위한 PyTorch 기반 라이브러리)
from monai.losses import DiceCELoss  # Dice + CrossEntropy 복합 손실 함수 (Segmentation에 특화)
from monai.inferers import sliding_window_inference  # 큰 3D 볼륨을 작은 슬라이딩 윈도로 나눠 추론할 때 사용
from monai.transforms import (  # 데이터 전처리 및 augmentation에 사용하는 다양한 변환들
    AsDiscrete,  # 모델 출력 결과를 이산형(one-hot 등)으로 변환
    Compose,  # 여러 변환을 순차적으로 묶는 파이프라인
    CropForegroundd,  # 관심 영역(FG)만 잘라내는 전처리
    LoadImaged,  # 이미지(.nii.gz 등) 파일을 메모리로 불러옴
    Orientationd,  # 의료 영상의 공간 방향을 표준화
    RandFlipd, RandRotate90d,  # 데이터 증강: 무작위 플립/회전
    RandCropByPosNegLabeld,  # 양성/음성 위치 기반 무작위 크롭 (불균형 데이터 보정)
    RandShiftIntensityd,  # 밝기 무작위 이동 (CT/MRI의 contrast 다양화)
    ScaleIntensityRanged,  # 픽셀 값 정규화 (일정 범위로 스케일링)
    Spacingd,  # 픽셀 간 간격(Spacing)을 표준화
    EnsureTyped,  # 데이터 타입 보정 (Tensor로 변환 등)
)

from monai.config import print_config  # MONAI, PyTorch, CUDA 등 환경 정보 출력용
from monai.metrics import DiceMetric  # Segmentation 모델 평가 지표 (Dice Score)
from monai.networks.nets import SwinUNETR  # 논문에서 사용하는 Swin UNETR 모델 클래스

# 📂 데이터셋 처리 관련 모듈
from monai.data import (
    ThreadDataLoader,  # PyTorch DataLoader의 병렬 처리 버전
    CacheDataset,  # 데이터셋을 메모리에 캐싱하여 빠른 접근 가능
    load_decathlon_datalist,  # JSON 포맷의 의료 영상 목록 로더
    decollate_batch,  # 배치 데이터를 개별 샘플로 나누는 유틸리티
    set_track_meta,  # 메타데이터 추적 여부 설정
)

# ⚙️ PyTorch 기본 모듈 (GPU 사용 및 텐서 연산)
import torch

# 💡 현재 설치된 MONAI, PyTorch, 기타 설정 정보 출력
print_config()

MONAI version: 1.6.dev2536
Numpy version: 2.3.1
Pytorch version: 2.6.0
MONAI flags: HAS_EXT = False, USE_COMPILED = False, USE_META_DICT = False
MONAI rev id: 5a50bb444bdb59d777e697b87248508b7b256efa
MONAI __file__: /Users/<username>/miniconda3/lib/python3.12/site-packages/monai/__init__.py

Optional dependencies:
Pytorch Ignite version: NOT INSTALLED or UNKNOWN VERSION.
ITK version: NOT INSTALLED or UNKNOWN VERSION.
Nibabel version: 5.3.2
scikit-image version: 0.25.2
scipy version: 1.15.2
Pillow version: 11.3.0
Tensorboard version: 2.18.0
gdown version: 5.2.0
TorchVision version: 0.21.0
tqdm version: 4.66.4
lmdb version: NOT INSTALLED or UNKNOWN VERSION.
psutil version: 7.0.0
pandas version: 2.3.1
einops version: 0.8.1
transformers version: 4.56.1
mlflow version: NOT INSTALLED or UNKNOWN VERSION.
pynrrd version: NOT INSTALLED or UNKNOWN VERSION.
clearml version: NOT INSTALLED or UNKNOWN VERSION.

For details about installing the optional dependencies, please visit:
    https://docs.mo

# 셀3: 데이터 저장 경로 설정
- 데이터가 저장될 경로를 설정하는 셀
- 환경 변수 MONAI_DATA_DIRECTORY가 있으면 해당 경로를 없으면 tempfile을 통해 임시 디렉토리 생성

In [3]:
# 환경 변수 'MONAI_DATA_DIRECTORY'가 설정되어 있는지 확인
# 이 환경변수가 있다면 그 위치에 데이터를 저장하고, 없다면 임시 폴더를 생성함
directory = os.environ.get("MONAI_DATA_DIRECTORY")

# 만약 MONAI_DATA_DIRECTORY가 설정되어 있다면 해당 디렉토리를 생성 (이미 존재하면 무시)
if directory is not None:
    os.makedirs(directory, exist_ok=True)

# 최종 데이터 저장 경로 결정
# - 환경변수로 지정된 경로가 있으면 그대로 사용
# - 없으면 tempfile을 이용해 임시 디렉토리를 생성
root_dir = tempfile.mkdtemp() if directory is None else directory

# 최종 저장 경로 출력
print(root_dir)

/var/folders/ss/b36xmfg949g_slyf613lmvhm0000gn/T/tmpu5hly7vk


# 셀4: 데이터 전처리 및 augmentation 정의
- 학습용과 검증용 데이터에 대해 각각의 변환 파이프라인 정의
- CropForeground, Orientation, Spacing, RandFlip, RandRotate90, RandShiftIntensity 등 MONAI에서 제공하는 3D 데이터 전처리 및 augmentation 기법들을 적용
- ScaleIntensityRanged: CT의 HU 값을 0~1 사이로 정규화

In [ ]:
# 학습 시 사용할 무작위 샘플 수 설정 (Crop 시 사용됨)
num_samples = 4

# CUDA 디바이스 설정 (Colab이나 여러 GPU 환경에서 디바이스 순서 보장)
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_transforms = Compose([
    # "image"와 "label" 키에 해당하는 데이터를 불러오고, 채널 차원을 추가함
    LoadImaged(keys=["image", "label"], ensure_channel_first=True),

    # CT 이미지의 HU 값을 -175 ~ 250 범위로 잘라낸 후, 0~1 사이로 정규화
    ScaleIntensityRanged(
        keys=["image"],
        a_min=-175,
        a_max=250,
        b_min=0.0,
        b_max=1.0,
        clip=True,
    ),

    # 이미지에서 foreground(장기가 있는 부분)만 잘라냄
    CropForegroundd(keys=["image", "label"], source_key="image"),

    # 의료 영상의 방향(Axis code)을 RAS(Right-Anterior-Superior) 표준으로 맞춤
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    # 픽셀 간 거리(spacing)를 표준화 → 모든 데이터 해상도 동일
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),  # 실제 거리 기준 (mm)
        mode=("bilinear", "nearest"),  # image는 bilinear, label은 nearest로 보간
    ),
    # 데이터를 GPU로 옮기고, 텐서로 변환함
    EnsureTyped(keys=["image", "label"], device=device, track_meta=False),

    # 랜덤 크롭 (장기 포함된 위치/아닌 위치를 모두 샘플링)
    RandCropByPosNegLabeld(
        keys=["image", "label"],
        label_key="label",
        spatial_size=(96, 96, 96),
        pos=1,
        neg=1,
        num_samples=num_samples,
        image_key="image",
        image_threshold=0,
    ),

    # 데이터 증강: 3축 각각에 대해 무작위로 좌우반전 (10% 확률)
    RandFlipd(keys=["image", "label"], spatial_axis=[0], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[1], prob=0.10),
    RandFlipd(keys=["image", "label"], spatial_axis=[2], prob=0.10),

    # 무작위 90도 회전 (최대 3회까지) → orientation 다양화
    RandRotate90d(keys=["image", "label"], prob=0.10, max_k=3),

    # 밝기 이동 (intensity shift): 50% 확률로 값 변경
    RandShiftIntensityd(keys=["image"], offsets=0.10, prob=0.50),
])


val_transforms = Compose([
    # 이미지/라벨 불러오기 (채널 추가 포함)
    LoadImaged(keys=["image", "label"], ensure_channel_first=True),

    # HU 정규화 (학습과 동일)
    ScaleIntensityRanged(
        keys=["image"], a_min=-175, a_max=250, b_min=0.0, b_max=1.0, clip=True
    ),

    # foreground 기준으로 잘라내기
    CropForegroundd(keys=["image", "label"], source_key="image"),

    # 방향 표준화
    Orientationd(keys=["image", "label"], axcodes="RAS"),

    # Spacing 통일
    Spacingd(
        keys=["image", "label"],
        pixdim=(1.5, 1.5, 2.0),
        mode=("bilinear", "nearest"),
    ),

    # GPU로 텐서 변환 (검증은 메타정보 추적 유지 → 시각화 등 가능)
    EnsureTyped(keys=["image", "label"], device=device, track_meta=True),
])

# 셀5: 데이터 디렉토리 생성
- 데이터를 다운로드나 저장을 위한 디렉토리 생성
- Linux 기반 셀에서 디렉토리 생성 명령어를 실행

In [ ]:
# Linux 명령어를 실행하는 셀 (Jupyter/Colab에서 사용 가능)
# !는 파이썬이 아닌 터미널 명령어(shell command)를 실행한다는 의미

# "data1"이라는 이름의 폴더를 생성함
# -p 옵션은 이미 해당 폴더가 존재하더라도 에러를 발생시키지 않음
# 즉, 폴더가 없으면 만들고, 있으면 넘어감
!mkdir -p data1

# 셀6: BTCV 데이터 압축 해제 및 정리
- BTCV 데이터셋이 압축된 형태(RawData.zip)로 제공될 경우를 대비한 처리
- Training/img와 Traning/label 하위 폴더에서 .nii.gz 파일을 각각 imagesTr, labelsTr 폴더로 이동
- 불필요한 RawData 폴더는 마지막에 삭제하여 깔끔한 디렉토리 구조 유지

In [ ]:
import zipfile
import os
import shutil

# 1. 압축 파일과 경로 설정
zip_path = "RawData.zip" # 압축 파일 이름
base_data_path = "./data1" # 전체 데이터를 저장할 기본 경로
extract_path = os.path.join(base_data_path, "RawData") # 압축을 풀 경로 설정

# 2. RawData.zip 압축 해제
# - 압축츨 풀면 일반적으로 RawData 폴더가 생성됨
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(extract_path)

# 3. 압축 해제된 폴더 안에 RawData 폴더가 중첩되어 있을 경우 처리
# - 예: ./data1/RawData/RawData/... 이런 구조일 경우 대비
inner_rawdata_path = os.path.join(extract_path, "RawData")
if os.path.exists(inner_rawdata_path):
    extract_path = inner_rawdata_path  # 실제 데이터 경로로 교체

# 4. 학습용 이미지/라벨 저장 폴더 생성 (imageTr / labelsTr)
imagesTr_path = os.path.join(base_data_path, "imagesTr")
labelsTr_path = os.path.join(base_data_path, "labelsTr")

os.makedirs(imagesTr_path, exist_ok=True)
os.makedirs(labelsTr_path, exist_ok=True)

# 5. 원본 이미지 및 라벨 폴더 경로 설정
img_src_dir = os.path.join(extract_path, "Training", "img")
label_src_dir = os.path.join(extract_path, "Training", "label")

# 6. .nil.gz 확장자를 가진 의료 영상 파일만 각각 해당 폴더로 이동
for filename in os.listdir(img_src_dir):
    if filename.endswith(".nii.gz"):
        shutil.move(
            os.path.join(img_src_dir, filename), os.path.join(imagesTr_path, filename)
        )

for filename in os.listdir(label_src_dir):
    if filename.endswith(".nii.gz"):
        shutil.move(
            os.path.join(label_src_dir, filename), os.path.join(labelsTr_path, filename)
        )


# 7. 불필요한 RawData 폴더 삭제 (압축파일까지 삭제 하고 싶으면 주석해제)
# os.remove(zip_path)  # 압축파일도 같이 삭제하려면 이 줄을 사용
shutil.rmtree(os.path.join(base_data_path, "RawData")) # 압축 풀고 남은 폴더 제거

# 완료 메시지 출력
print("완료! 모든 파일 이동 및 정리 완료")

# 셀7: 학습/검증 JSON 설정 및 파일 다운로드
- MONAI 튜토리얼용으로 제공되는 BTCV용 split 파일 다운로드
- 학습 및 검증 데이터셋을 JSON 포맷으로 정의한 파일 (dataset_0.json)

In [ ]:
# ⬇️ wget: 웹에서 파일을 다운로드하는 명령어
# -O 옵션: 저장할 파일명을 직접 지정함

# ✅ NVIDIA MONAI 튜토리얼에서 제공하는 BTCV 학습/검증용 JSON 설정 파일을 다운로드함
# 이 JSON에는 학습/검증 데이터 리스트, 파일 경로, 분할 정보 등이 담겨 있음
!wget -O data1/dataset_0.json 'https://developer.download.nvidia.com/assets/Clara/monai/tutorials/swin_unetr_btcv_dataset_0.json'

# 셀8: 데이터셋 로드 및 DataLoader 생성
- JSON에 정의된 학습/검증 데이터를 load_decathlon_datalist()로 로딩
- CacheDataset을 사용하여 학습 데이터를 캐싱하여 빠르게 로드 가능
- 학습/검증용 DataLoader 생성 (ThreadDataLoader 사용)
- set_track_meta(False)를 통해 학습 이미지에 메타데이터를 추적하지 않도록 설정

In [ ]:
# ✅ 데이터셋이 저장된 폴더 경로 및 JSON split 파일 이름 설정
data_dir = "data1/"
split_json = "dataset_0.json"

# ✅ JSON 파일의 전체 경로 구성
datasets = data_dir + split_json   # "data1/dataset_0.json"

# ✅ 학습/검증용 파일 리스트 로딩
# - JSON 내의 "training" / "validation" 항목을 각각 불러옴
datalist = load_decathlon_datalist(datasets, True, "training")    # 학습용 이미지 목록
val_files = load_decathlon_datalist(datasets, True, "validation") # 검증용 이미지 목록

# ✅ 학습용 Dataset 정의 (전처리 포함)
train_ds = CacheDataset(
    data=datalist,                  # 학습용 이미지 목록
    transform=train_transforms,     # 이전에 정의한 전처리 파이프라인
    cache_num=24,                   # 최대 24개 샘플까지 캐싱 (메모리에 저장)
    cache_rate=1.0,                 # 전체 데이터를 캐싱
    num_workers=8,                  # 백그라운드에서 데이터를 로드할 워커 수
)

# ✅ 학습용 DataLoader 정의
train_loader = ThreadDataLoader(
    train_ds,
    num_workers=0,                  # ThreadDataLoader는 멀티프로세싱 대신 멀티스레드 사용
    batch_size=1,
    shuffle=True                    # 학습 데이터는 매 epoch마다 섞음
)

# ✅ 검증용 Dataset 정의
val_ds = CacheDataset(
    data=val_files,
    transform=val_transforms,   # 검증용 전처리 파이프라인
    cache_num=6,
    cache_rate=1.0,
    num_workers=4,
)

# ✅ 검증용 DataLoader 정의
val_loader = ThreadDataLoader(
    val_ds,
    num_workers=0,
    batch_size=1,               # 배치 크기 1 (3D 의료 영상은 메모리 부담 큼)
)

# ✅ EnsureTyped()로 개별 샘플의 MetaTensor 여부를 제어할 수 있지만,
# 전역적으로도 set_track_meta(False)를 설정하여 학습용 데이터는 텐서로만 처리되게 함
set_track_meta(False)

# 셀9: 데이터 로딩 확인 (결함 여부 체크)
- LoadImage()로 각 이미지가 제대로 로드되는지 테스트
- 오류가 발생하면 해당 파일명을 출력하며 검사 중단
- 데이터셋을 사용하기 전 데이터 손상 여부를 사전 점검하는 셀

In [ ]:
# ✅ LoadImage는 단일 이미지(.nii, .nii.gz 등)를 불러오는 MONAI 전처리 클래스
from monai.transforms import LoadImage

loader = LoadImage()          # 이미지 로더 초기화
error_found = False           # 에러 감지 여부를 기록할 플래그

# ✅ datalist와 val_files를 하나로 합침
all_data = datalist + val_files

# ✅ 전체 이미지 로딩 확인
for i, item in enumerate(all_data):
    try:
        print(f"[{i+1}/{len(all_data)}] Checking {item['image']}")  # 현재 체크 중인 이미지 경로 출력
        img = loader(item["image"])                                  # 실제 이미지 로드 시도
    except Exception as e:
        print(f"❌ Failed to load {item['image']}")                  # 에러 발생 시 경고 출력
        print(e)                                                     # 에러 상세 정보 출력
        error_found = True
        break

# ✅ 모든 이미지가 정상적으로 로드된 경우, 최종 메시지 출력
if not error_found:
    print("모든 이미지가 정상적으로 로드되었습니다! 😊")

# 셀10: 시각화 - 이미지와 레벨 확인
- slice_map을 통해 적절한 z-출 슬라이스 인덱스를 정의
- plt.imshow()를 사용해 CT 슬라이스와 라벨을 나란히 시각화
- 데이터 구조 확인 및 시각적으로 품질 확인 가능